# B1: Weight tour

Before writing any model code we look at exactly what is inside the checkpoint: every tensor's name and shape, where the 1.1B parameters sit, and what that means for memory.

Our own model (Part B) must load these tensors **by name**, so `tinyserve/weights.py` holds a pure-Python *spec* of every name and shape. Here we check the spec against the real model.

Questions this lesson answers:
1. What are the 201 tensors?
2. Which part of the model holds most of the bytes (and therefore most of each decode step's memory traffic, Lesson A2)?
3. What does grouped-query attention (GQA) change in the shapes, and what does the KV cache cost per token?

In [ ]:
# Setup: clone or hard-reset the repo to GitHub, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    # Colab may hold untracked files (e.g. results written by a notebook) that clash with a pull.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import importlib, json
import torch
import tinyserve.basics, tinyserve.quiet, tinyserve.weights
for module in (tinyserve.quiet, tinyserve.basics, tinyserve.weights):
    importlib.reload(module)  # picks up repo edits without restarting the kernel
from tinyserve.basics import load_model
from tinyserve.envinfo import collect_env
from tinyserve import weights

print(collect_env())
tokenizer, model = load_model(device="cuda", dtype=torch.float16)
config = model.config
print(f"layers {config.num_hidden_layers}, hidden {config.hidden_size}, MLP width {config.intermediate_size}, "
      f"query heads {config.num_attention_heads}, KV heads {config.num_key_value_heads}, head_dim {weights.head_dim(config)}, "
      f"vocab {config.vocab_size}, tied embeddings {config.tie_word_embeddings}")

## 1. The tensors
A Llama checkpoint is a flat dictionary `name -> tensor`. There is one tensor for the token embedding, nine per decoder layer, a final norm, and an `lm_head`. With 22 layers that is `1 + 9*22 + 1 + 1 = 201`.

In [ ]:
rows = weights.weight_tour(model.named_parameters())
print("tensors:", len(rows), "| dtypes:", {row["dtype"] for row in rows})
print()
for row in rows[:1] + rows[1:10] + rows[-2:]:
    print(f"{row['name']:<55} {str(row['shape']):>14} {row['numel']:>12,}")

## 2. Spec vs reality
`expected_shapes(config)` builds the same dictionary from the config alone. If any name is missing, unexpected, or has a different shape, `compare_shapes` tells us. B8 will rely on this spec.

In [ ]:
actual_shapes = {row["name"]: row["shape"] for row in rows}
diff = weights.compare_shapes(actual_shapes, weights.expected_shapes(config))
print("differences:", diff)
assert diff == {"missing": [], "unexpected": [], "wrong_shape": []}

actual_total = sum(row["numel"] for row in rows)
print(f"parameters: actual {actual_total:,}  spec {weights.expected_param_count(config):,}")
assert actual_total == weights.expected_param_count(config)

layer0 = sum(row["numel"] for row in rows if row["name"].startswith("model.layers.0."))
print(f"one decoder layer: {layer0:,} parameters  x {config.num_hidden_layers} layers = {layer0 * config.num_hidden_layers:,}")

## 3. Where the bytes are
In a decode step every weight is read once (Lesson A2), so the share of bytes is also the share of memory traffic.

In [ ]:
totals = weights.bytes_by_category(rows)
for name, entry in sorted(totals.items(), key=lambda item: -item[1]["nbytes"]):
    print(f"{name:<10} {entry['numel'] / 1e6:9.1f} M params  {entry['nbytes'] / 1e6:9.1f} MB  {entry['share']:6.1%}")
print(f"{'total':<10} {sum(e['numel'] for e in totals.values()) / 1e6:9.1f} M params  {sum(e['nbytes'] for e in totals.values()) / 1e6:9.1f} MB")

## 4. GQA and the KV cache
The 32 query heads share only 4 key/value heads (8 queries per KV head). So `k_proj` and `v_proj` output `4 x 64 = 256` features instead of 2048. That makes those projections 8x smaller, and, more importantly, the **KV cache 8x smaller**: for every token we store K and V for each layer.

`KV bytes per token = 2 (K and V) x layers x KV heads x head_dim x bytes per element`.

In [ ]:
per_token = weights.kv_cache_bytes_per_token(config, bytes_per_element=2)
mha_per_token = 2 * config.num_hidden_layers * config.num_attention_heads * weights.head_dim(config) * 2
print(f"KV cache per token (FP16): GQA {per_token:,} bytes = {per_token / 1024:.0f} KiB; full multi-head would be {mha_per_token:,} bytes = {mha_per_token / 1024:.0f} KiB")
for context in (128, 2048):
    print(f"  one sequence of {context:>4} tokens: {per_token * context / 2**20:7.1f} MiB   (weights: {sum(r['nbytes'] for r in rows) / 2**20:.0f} MiB)")
print(f"sequences of 2048 tokens that fit in 10 GiB of free memory: {int(10 * 2**30 // (per_token * 2048))}")

## 5. What the weights look like
Statistics of a few real tensors (computed in FP32). Norm weights are scales around 1 or smaller; projection weights are small numbers centred on 0.

In [ ]:
params = dict(model.named_parameters())
stats_rows = {}
for name in ("model.embed_tokens.weight", "model.layers.0.self_attn.q_proj.weight", "model.layers.0.mlp.down_proj.weight",
             "model.layers.0.input_layernorm.weight", "model.layers.21.input_layernorm.weight", "model.norm.weight", "lm_head.weight"):
    stats = weights.tensor_stats(params[name])
    stats_rows[name] = stats
    print(f"{name:<45} mean {stats['mean']:8.4f}  std {stats['std']:7.4f}  min {stats['min']:8.3f}  max {stats['max']:8.3f}")

## 6. Save results and run the tests

In [ ]:
from tinyserve.envinfo import save_result

saved_path = save_result("B1_weights", {
    "num_tensors": len(rows),
    "num_params": actual_total,
    "params_per_layer": layer0,
    "by_category": totals,
    "kv_bytes_per_token_fp16": per_token,
    "kv_bytes_per_token_fp16_if_mha": mha_per_token,
    "tensor_stats": stats_rows,
    "layer0_shapes": {row["name"]: row["shape"] for row in rows if row["name"].startswith("model.layers.0.")},
})
print(saved_path)

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b1_weights.py", "tests/test_quiet.py", "-v"], capture_output=True, text=True)
print(result.stdout[-3500:], result.stderr[-1000:])

In [ ]:
# Print the full results file so it can be copied into the repo from the saved notebook.
print(open(saved_path).read())